# TP 1 - Prompt Engineering

Ce notebook introduit les bases de l'interaction avec un LLM via l'API Google GenAI.
On construit progressivement un assistant de voyage, du prompt brut jusqu'à la sortie structurée.


### 0.1. Documentation générale des librairies utilisées

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

### 0.2. Le format Notebook (.ipynb)

Un fichier `.ipynb` est un document interactif composé de **cellules** que l'on exécute une à une, dans l'ordre.

Il existe deux types de cellules :
- **Cellule code** : contient du code Python exécutable
- **Cellule markdown** : contient du texte formaté (titres, listes, liens…)

Exécutez chaque cellule avec `Shift+Enter` ou le bouton ▶ dans la barre d'outils.

In [ ]:
import json

from pydantic import BaseModel

from shared.config import ROOT_DIR
from shared.misc_utils import write_json_file

# INFO : Choix entre local ou cloud (LLM)
from shared.llm_utils import (
    LLMRequest,
    run_llm, # Cloud
    #run_llm_local as run_llm, # Local
    run_llm_structured, # Cloud
    #run_llm_local_structured as run_llm_structured, # Local
)

LOG_DIR = ROOT_DIR / "TP1_travel_planner_LLM" / "logs"

### 0.3. Use case principal
Votre objectif est de répondre à une question

In [ ]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire de voyage. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

#### Fonctions auxiliaires pour calculer le coût et afficher l'usage des tokens

In [2]:
def token_estimate_cost_usd(
    token_usage: dict[str, int],
    input_price_per_1m_tokens_usd: float = 0.30,
    output_price_per_1m_tokens_usd: float = 2.50,
) -> float:
    input_cost = token_usage["input_tokens"] * input_price_per_1m_tokens_usd / 1_000_000
    output_cost = token_usage["output_tokens"] * output_price_per_1m_tokens_usd / 1_000_000
    return input_cost + output_cost


def token_print_report(token_usage: dict[str, int]) -> None:
    estimated_cost = token_estimate_cost_usd(token_usage)
    print(
        f"tokens : entrée={token_usage['input_tokens']} | "
        f"sortie={token_usage['output_tokens']} | "
        f"total={token_usage['total_tokens']}"
    )
    print(f"coût estimé (USD) : {estimated_cost:.6f}")


### 0.4. Récapitulatif des fonctions utilisées dans ce notebook

**Fonctions et classes à utiliser**

- `project_settings` : objet qui centralise la configuration partagée du modèle (température, top_p, top_k, max_tokens)
- `genai_client` : client qui s'authentifie à l'API Google GenAI, créé une seule fois

--> Disponibles dans `shared/config.py`

- `LLMRequest` : classe qui représente les données d'entrée d'un appel LLM (`user_prompt` obligatoire, `system_prompt` optionnel)
- `LLMResponse` : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)
- `run_llm` : envoie une requête en texte libre et retourne un `LLMResponse`
- `run_llm_structured` : comme `run_llm`, mais impose une sortie JSON conforme à un `response_schema`

--> Disponibles dans `shared/llm_utils.py`

---
## 1. Prompt simple

Sans system prompt, le modèle s'appuie uniquement sur son instruction générale de base et choisit librement le style et la structure de sa réponse.

**Attention à bien séparer la requête utilisateur et les tâches du LLM.**

Ici, par exemple :
- La requête utilisateur porte sur une ville, des envies et des contraintes spécifiques
- Le system prompt décrit le rôle du modèle (agent de voyage) et la forme attendue de la réponse (structure, style, format).

--> Le system prompt est réutilisable : **il ne dépend pas d'un cas d'usage précis**.

Il faut donc trouver un bon équilibre entre prompt généraliste (réutilisable) et prompt spécifique (adapté à la requête utilisateur)

Dans ce cas, le prompt système doit définir le **rôle d'agent de voyage**, poser des contraintes de contenu et de **style**, imposer une **structure** de réponse claire, et vérifier que les **contraintes utilisateur** (budget, temps) sont valides.

### 1.1. Construire la requête et appeler le LLM

In [ ]:
request_v1 = LLMRequest(
    system_prompt=None,
    user_prompt=user_query,
)
run_result_v1 = await run_llm(request_v1)
final_text_v1 = run_result_v1.output

token_usage_v1 = {
    "input_tokens": int(run_result_v1.input_tokens),
    "output_tokens": int(run_result_v1.output_tokens),
    "total_tokens": int(run_result_v1.total_tokens),
}

log_path_v1 = LOG_DIR / "llm_output_v1.txt"
write_json_file(file_path=log_path_v1, data=run_result_v1.raw_response)

print(final_text_v1)

### 1.2. Afficher l'usage des tokens

In [4]:
token_print_report(token_usage_v1)


tokens : entrée=69 | sortie=3359 | total=4193
coût estimé (USD) : 0.008418


---
## 2. System prompt structuré

Sans system prompt, le modèle choisit librement sa structure de réponse.

--> Un system prompt permet d'imposer un rôle, des contraintes et un format de sortie cohérent.

Pour améliorer la réponse, définir dans `system_prompt_v2` :
- un **rôle** explicite pour le modèle
- des **contraintes** de contenu et de style
- une **structure** en **4 sections** : résumé, itinéraire, budget, conseils

### 2.1. Rédiger le system prompt

In [ ]:
system_prompt_v2 = """
## Instructions
Tu es un assistant expert en planification de voyage.
Réponds avec un texte structuré, clair et utile immédiatement.

Contraintes à respecter :
- Respecter la durée demandée.
- Respecter le budget indiqué.

Contraintes de style :
- Aller à l'essentiel, sans phrases inutiles.
- Pas de Markdown décoratif (gras, mise en forme complexe).
- Tu peux utiliser '##' pour séparer les sections.
- Pour chaque recommandation, donner 1 raison courte + 1 détail pratique concret.

Structure de réponse obligatoire :
1) Résumé
- Réponse directe en 2 à 4 phrases.
2) Itinéraire
- Plan jour par jour avec activités matin/après-midi.
- Inclure recommandations déjeuner/dîner.
3) Budget
- Estimation par catégorie (activités, repas, transport, extras).
4) Conseils pratiques
- Donner 3 conseils actionnables et pertinents.

Notes complémentaires :
- Si les dates sont flexibles, proposer la période la plus adaptée.
- Si le budget est serré, proposer une alternative moins chère pour chaque poste coûteux.
"""

### 2.2. Construire la requête et appeler le LLM

In [ ]:
request_v2 = LLMRequest(system_prompt=system_prompt_v2, user_prompt=user_query)
run_result_v2 = await run_llm(request_v2)
final_text_v2 = run_result_v2.output

token_usage_v2 = {
    "input_tokens": int(run_result_v2.input_tokens),
    "output_tokens": int(run_result_v2.output_tokens),
    "total_tokens": int(run_result_v2.total_tokens),
}

log_path_v2 = LOG_DIR / "llm_output_v2.txt"
write_json_file(file_path=log_path_v2, data=run_result_v2.raw_response)

print(final_text_v2)

### 2.3. Afficher l'usage des tokens

In [6]:
token_print_report(token_usage_v2)


tokens : entrée=322 | sortie=1261 | total=2328
coût estimé (USD) : 0.003249


---
## 3. Sortie structurée (Structured Output)

On pourrait demander dans le prompt au LLM de suivre un format JSON (« réponds avec un JSON comme ceci : ... »), ce qui est fait souvent en pratique, mais reste peu fiable et complexifie le parsing.

Gemini propose une solution plus propre : la **sortie structurée** (*structured output*). Avec `response_schema` (un modèle Pydantic) et `response_mime_type="application/json"`, l'API garantit un JSON valide conforme au schéma, plus besoin de parsing.

Doc : https://ai.google.dev/gemini-api/docs/structured-output

### 3.1. Définir la structure de sortie

In [ ]:
class ActivityItem(BaseModel):
    title: str
    address: str
    estimated_cost_eur: float


class MealItem(BaseModel):
    name: str
    address: str
    estimated_cost_eur: float


class DayPlan(BaseModel):
    day: str  # ex: "day_1"
    activity_am: ActivityItem
    activity_pm: ActivityItem
    lunch: MealItem
    dinner: MealItem


class TravelAgenda(BaseModel):
    # DOC (additionalProperties) : dict[str, ...] genere un schema JSON avec
    # "additionalProperties", non supporte par l'API Gemini pour response_schema.
    # https://ai.google.dev/gemini-api/docs/structured-output#json-schemas
    # -> on utilise une liste avec un champ "day" plutot qu'un dict indexe par jour.
    agenda: list[DayPlan]

### 3.2. Construire la requête et appeler le LLM

In [ ]:
# system_prompt_v3 reprend system_prompt_v2 : response_schema (via run_llm_structured) garantit
# déjà le format de sortie, plus besoin de le décrire dans le prompt.
system_prompt_v3 = system_prompt_v2

request_v3 = LLMRequest(system_prompt=system_prompt_v3, user_prompt=user_query)
run_result_v3 = await run_llm_structured(request_v3, response_schema=TravelAgenda)
final_text_v3 = run_result_v3.output

token_usage_v3 = {
    "input_tokens": int(run_result_v3.input_tokens),
    "output_tokens": int(run_result_v3.output_tokens),
    "total_tokens": int(run_result_v3.total_tokens),
}

log_path_v3 = LOG_DIR / "llm_output_v3.txt"
write_json_file(file_path=log_path_v3, data=run_result_v3.raw_response)

print(final_text_v3)

### 3.3. Afficher l'usage des tokens

In [8]:
token_print_report(token_usage_v3)


tokens : entrée=322 | sortie=943 | total=2018
coût estimé (USD) : 0.002454


### 3.4. Parser le JSON

`response_schema` garantit que `final_text_v3` est déjà un JSON valide conforme au schéma : plus besoin de fonction de parsing maison, un simple `json.loads` suffit.

In [ ]:
parsed_json_v3 = json.loads(final_text_v3)
print(json.dumps(obj=parsed_json_v3, ensure_ascii=False, indent=2))

### 3.5. Vérifier le budget

Finalement, on peut traiter le JSON parsé pour calculer le coût total estimé des activités proposées, et vérifier que cela respecte la contrainte de budget donnée dans la requête utilisateur.

In [10]:
agenda_v3 = parsed_json_v3["agenda"]

def compute_total_cost_from_json(agenda: list[dict[str, object]]) -> float:
    total_cost = 0.0
    for day in agenda:
        for slot_name in ["activity_am", "activity_pm", "lunch", "dinner"]:
            total_cost += float(day[slot_name]["estimated_cost_eur"])
    return total_cost

total_cost_eur = compute_total_cost_from_json(agenda_v3)
average_per_day = total_cost_eur / len(agenda_v3)

print(f"Total estimé : {total_cost_eur:.2f} EUR")
print(f"Moyenne par jour : {average_per_day:.2f} EUR")


Total estimé : 227.00 EUR
Moyenne par jour : 56.75 EUR
